# Phase 5 — Activation Patching: Causal Localization of Emotions

**Causal intervention** to find where each emotion's information lives inside BERT.

Lesion studies (notebook 08) show what *breaks* when a layer is compressed, but
cannot distinguish between two hypotheses:

1. The layer **computes** critical information (source).
2. The layer merely **transmits** information computed elsewhere (path).

Activation patching (Meng et al., 2022) resolves the ambiguity:
1. Build an aggressively compressed model (rank 64) with degraded F1.
2. For each encoder layer L (0-11), run the compressed model but **replace** the
   output of L with the baseline activation captured for the same input.
3. Measure how much per-emotion F1 is **restored** by this single-layer patch.

The layer whose patching restores the most F1 is where compression destroyed the
emotion-critical information.

**Pipeline.**
- **Setup** — paths, imports, helpers (dark theme kept for figure parity).
- **Part A** — load baseline + compressed model, compute reference F1s.
- **Part B** — layer-level patching (12 layers).
- **Part C** — component-level patching (attention vs FFN on top-K layers).
- **Part D** — per-emotion causal profiles + brain scan.
- **Part E** — cross-validation against the lesion study.
- **Part F** — export every result as CSV to `results/notebook5/`.

## Setup

In [ ]:
import sys, os, time, gc, warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.utils import compute_metrics
from src.compression import apply_svd_compression, get_target_layer_names

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
EXPORT_DIR = os.path.join(RESULTS_DIR, 'notebook5')
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_SUBDIR = 'bert-goemotions-23emo-final'
MODEL_PATH = os.path.join(RESULTS_DIR, MODEL_SUBDIR)

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']
PATCH_RANK = 64      # aggressive compression for clear patching signal
TOP_K_LAYERS = 4     # layers for component-level patching
VALID_F1_THRESHOLD = 0.01
GAP_EPSILON = 0.005  # avoid divide-by-zero in restoration score
BATCH_SIZE = 64
N_ENCODER_LAYERS = 12

DARK_BG        = '#0d1117'
DARK_GRID      = '#21262d'
TEXT_PRIMARY   = '#e6edf3'
TEXT_SECONDARY = '#8b949e'

NEON_CMAP = LinearSegmentedColormap.from_list(
    'neon', ['#00ff87', '#00d4ff', '#7b2ff7', '#ff006e'], N=256)
RESTORATION_CMAP = LinearSegmentedColormap.from_list(
    'restoration', ['#0d1117', '#1a1a2e', '#16213e', '#0f3460',
                    '#00d4ff', '#00ff87', '#ffd60a', '#ff006e'], N=256)

plt.rcParams.update({
    'figure.facecolor':  DARK_BG,
    'axes.facecolor':    DARK_BG,
    'text.color':        TEXT_PRIMARY,
    'axes.labelcolor':   TEXT_PRIMARY,
    'xtick.color':       TEXT_SECONDARY,
    'ytick.color':       TEXT_SECONDARY,
    'axes.edgecolor':    DARK_GRID,
    'grid.color':        DARK_GRID,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.facecolor': DARK_BG,
    'savefig.bbox':      'tight',
    'font.size':         12,
})

print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
print(f'Export dir:   {EXPORT_DIR}')
print(f'Patch rank:   {PATCH_RANK}')

In [ ]:
@torch.no_grad()
def run_patched_inference(baseline_model, compressed_model, dataset, data_collator,
                          patch_layer_idx, patch_component='all', batch_size=BATCH_SIZE):
    """Run compressed model with activations patched from baseline at one layer.

    patch_component: 'all' (entire BertLayer output), 'attention', or 'ffn'.
    Returns (per_emotion_f1 (num_labels,), f1_macro scalar).
    """
    dev = next(baseline_model.parameters()).device
    all_logits, all_labels = [], []

    module_map = {
        'all':       lambda layer: layer,
        'attention': lambda layer: layer.attention,
        'ffn':       lambda layer: layer.output,
    }
    pick = module_map[patch_component]

    for i in range(0, len(dataset), batch_size):
        batch_indices = range(i, min(i + batch_size, len(dataset)))
        batch = data_collator([dataset[j] for j in batch_indices])
        input_ids = batch['input_ids'].to(dev)
        attention_mask = batch['attention_mask'].to(dev)

        cache = {}
        base_layer = pick(baseline_model.bert.encoder.layer[patch_layer_idx])
        comp_layer = pick(compressed_model.bert.encoder.layer[patch_layer_idx])

        handle = base_layer.register_forward_hook(lambda m, i, o: cache.__setitem__('out', o))
        baseline_model(input_ids=input_ids, attention_mask=attention_mask)
        handle.remove()

        handle = comp_layer.register_forward_hook(lambda m, i, o: cache['out'])
        outputs = compressed_model(input_ids=input_ids, attention_mask=attention_mask)
        handle.remove()

        all_logits.append(outputs.logits.cpu())
        all_labels.append(batch['labels'])

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    per_emotion_f1 = f1_score(labels, preds, average=None, zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    return per_emotion_f1, f1_macro


def compute_restoration(patched_f1, compressed_f1, baseline_f1, eps=GAP_EPSILON):
    """Restoration score = (patched - compressed) / (baseline - compressed), clipped to [0,1]."""
    gap = baseline_f1 - compressed_f1
    safe = np.where(np.abs(gap) > eps, gap, 1.0)
    rest = (patched_f1 - compressed_f1) / safe
    rest = np.clip(rest, 0.0, 1.0)
    no_gap = np.abs(gap) <= eps
    rest = np.where(no_gap, 0.0, rest)
    return rest

## Part A — Load Models and Establish Baselines

Load the fine-tuned BERT, build its rank-64 compressed copy, and evaluate both on
the test set to get per-emotion baseline / compressed F1.

In [ ]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, problem_type='multi_label_classification',
)
baseline_model.eval().to(DEVICE)
print(f'Baseline loaded from {MODEL_PATH}')

all_target_names = get_target_layer_names(baseline_model)
compressed_model = apply_svd_compression(baseline_model, rank=PATCH_RANK, layer_names=all_target_names)
compressed_model.eval().to(DEVICE)
print(f'Compressed model built (rank={PATCH_RANK}, {len(all_target_names)} layers)')

_, _, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)
num_labels = len(emotion_names)
print(f'Test set: {len(test_ds)} examples  |  {num_labels} emotions')

In [ ]:
eval_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'tmp_eval'),
    per_device_eval_batch_size=BATCH_SIZE,
    fp16=(DEVICE == 'cuda'),
    report_to='none',
)

def evaluate(model):
    trainer = Trainer(model=model, args=eval_args,
                      compute_metrics=compute_metrics, data_collator=data_collator)
    m = trainer.evaluate(test_ds)
    per_emo = np.array([m[f'eval_f1_label_{i}'] for i in range(num_labels)])
    return per_emo, m['eval_f1_macro']

print('Evaluating baseline...')
baseline_per_emotion, baseline_f1_macro = evaluate(baseline_model)
print('Evaluating compressed...')
compressed_per_emotion, compressed_f1_macro = evaluate(compressed_model)

print(f'\nBaseline   F1 macro: {baseline_f1_macro:.4f}')
print(f'Compressed F1 macro: {compressed_f1_macro:.4f}')
print(f'Gap:                 {baseline_f1_macro - compressed_f1_macro:.4f}')
print(f'Per-emotion F1 range:')
print(f'  Baseline:   [{baseline_per_emotion.min():.4f}, {baseline_per_emotion.max():.4f}]')
print(f'  Compressed: [{compressed_per_emotion.min():.4f}, {compressed_per_emotion.max():.4f}]')

valid_mask = baseline_per_emotion > VALID_F1_THRESHOLD
valid_indices = np.where(valid_mask)[0]
valid_emotions = [emotion_names[i] for i in valid_indices]
n_valid = len(valid_emotions)
print(f'\nValid emotions: {n_valid}  (excluded {num_labels - n_valid} with baseline F1 <= {VALID_F1_THRESHOLD})')

## Part B — Layer-level Patching (12 Layers)

In [ ]:
layer_patched_f1 = np.zeros((N_ENCODER_LAYERS, num_labels))
layer_patched_macro = np.zeros(N_ENCODER_LAYERS)

print(f'Layer-level activation patching (rank={PATCH_RANK})')
print('=' * 60)
t_total = time.time()

for layer_idx in range(N_ENCODER_LAYERS):
    t0 = time.time()
    per_emo, f1_m = run_patched_inference(
        baseline_model, compressed_model, test_ds, data_collator,
        patch_layer_idx=layer_idx, patch_component='all',
    )
    layer_patched_f1[layer_idx] = per_emo
    layer_patched_macro[layer_idx] = f1_m
    rest = (f1_m - compressed_f1_macro) / max(baseline_f1_macro - compressed_f1_macro, 1e-8)
    print(f'  Layer {layer_idx:2d}: F1 = {f1_m:.4f}  (restoration = {rest:.1%})  [{time.time()-t0:.1f}s]')

print(f'\nCompleted in {time.time() - t_total:.0f}s')
best_layer = int(np.argmax(layer_patched_macro))
print(f'Best single-layer patch: Layer {best_layer}  (F1 = {layer_patched_macro[best_layer]:.4f})')

### B.1 — Restoration Map (Heatmap)

In [ ]:
# Build restoration matrix: rows = layers, cols = emotions
restoration_matrix = np.zeros((N_ENCODER_LAYERS, num_labels))
for i in range(num_labels):
    restoration_matrix[:, i] = compute_restoration(
        layer_patched_f1[:, i], compressed_per_emotion[i], baseline_per_emotion[i]
    )

restoration_valid = restoration_matrix[:, valid_indices]
critical_layer_per_emotion = np.argmax(restoration_valid, axis=0)
sort_idx = np.argsort(critical_layer_per_emotion)
sorted_emotions = [valid_emotions[i] for i in sort_idx]
restoration_sorted = restoration_valid[:, sort_idx]

fig, ax = plt.subplots(figsize=(20, 10))
im = ax.imshow(restoration_sorted, cmap=RESTORATION_CMAP, vmin=0, vmax=1,
               aspect='auto', interpolation='nearest')
ax.set_yticks(range(N_ENCODER_LAYERS))
ax.set_yticklabels([f'Layer {i}' for i in range(N_ENCODER_LAYERS)], fontsize=12, fontweight='bold')
ax.set_xticks(range(len(sorted_emotions)))
ax.set_xticklabels(sorted_emotions, rotation=55, ha='right', fontsize=10)

for i in range(N_ENCODER_LAYERS):
    for j in range(len(sorted_emotions)):
        val = restoration_sorted[i, j]
        if val > 0.02:
            txt_color = '#000000' if val > 0.6 else TEXT_PRIMARY
            ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                    fontsize=6, color=txt_color, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Restoration Score', fontsize=13, color=TEXT_PRIMARY, labelpad=12)
cbar.ax.tick_params(colors=TEXT_SECONDARY, labelsize=11)
cbar.outline.set_edgecolor(DARK_GRID)

ax.set_title('CAUSAL RESTORATION MAP\nWhere Each Emotion\'s Information Lives',
             fontsize=22, fontweight='bold', pad=18,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
fig.text(0.45, 0.02,
         f'Activation patching: baseline activations injected into compressed model (rank {PATCH_RANK})  |  '
         f'Sorted by critical layer  |  {n_valid} emotions',
         fontsize=10, color=TEXT_SECONDARY, ha='center',
         path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

plt.savefig(os.path.join(RESULTS_DIR, 'activation_patching_01_restoration_map.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()

### B.2 — Mean Restoration per Layer

In [ ]:
mean_restoration_per_layer = restoration_valid.mean(axis=1)

fig, ax = plt.subplots(figsize=(14, 6))
bar_colors = [NEON_CMAP(v / mean_restoration_per_layer.max()) for v in mean_restoration_per_layer]
bars = ax.bar(range(N_ENCODER_LAYERS), mean_restoration_per_layer, color=bar_colors,
              edgecolor=DARK_GRID, linewidth=1.5, width=0.75)
for bar, val in zip(bars, mean_restoration_per_layer):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11,
            color=TEXT_PRIMARY, fontweight='bold')

ax.set_xlabel('Encoder Layer', fontsize=14, labelpad=10)
ax.set_ylabel('Mean Restoration Score', fontsize=14, labelpad=10)
ax.set_xticks(range(N_ENCODER_LAYERS))
ax.set_xticklabels([f'L{i}' for i in range(N_ENCODER_LAYERS)], fontsize=13, fontweight='bold')
ax.set_ylim(0, mean_restoration_per_layer.max() * 1.15)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')

ax.set_title('MEAN CAUSAL RESTORATION PER LAYER\n'
             'Which layers carry the most emotion-critical information?',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

plt.savefig(os.path.join(RESULTS_DIR, 'activation_patching_02_mean_restoration.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()

## Part C — Component-level Patching (Attention vs FFN)

For the top-K most impactful layers, patch only attention or only FFN to
localise within each layer.

In [ ]:
top_layers = np.argsort(mean_restoration_per_layer)[-TOP_K_LAYERS:][::-1]
print(f'Top {TOP_K_LAYERS} layers: {list(map(int, top_layers))}')
print(f'Mean restoration: {[f"{mean_restoration_per_layer[l]:.3f}" for l in top_layers]}')

component_patched = {}
print(f'\nComponent-level activation patching')
print('=' * 60)
t_total = time.time()

for layer_idx in top_layers:
    for component in ['attention', 'ffn']:
        t0 = time.time()
        per_emo, f1_m = run_patched_inference(
            baseline_model, compressed_model, test_ds, data_collator,
            patch_layer_idx=int(layer_idx), patch_component=component,
        )
        component_patched[(int(layer_idx), component)] = (per_emo, f1_m)
        rest = (f1_m - compressed_f1_macro) / max(baseline_f1_macro - compressed_f1_macro, 1e-8)
        print(f'  Layer {layer_idx:2d} [{component:9s}]: F1 = {f1_m:.4f}  '
              f'(restoration = {rest:.1%})  [{time.time()-t0:.1f}s]')

print(f'\nCompleted in {time.time() - t_total:.0f}s')
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

### C.1 — Full Layer vs Attention vs FFN

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(top_layers))
width = 0.25
macro_gap = max(baseline_f1_macro - compressed_f1_macro, 1e-8)

full_rest = [(layer_patched_macro[l] - compressed_f1_macro) / macro_gap for l in top_layers]
attn_rest = [(component_patched[(int(l), 'attention')][1] - compressed_f1_macro) / macro_gap for l in top_layers]
ffn_rest  = [(component_patched[(int(l), 'ffn')][1]       - compressed_f1_macro) / macro_gap for l in top_layers]

bars_sets = [
    (ax.bar(x - width, full_rest, width, label='Full Layer',     color='#00ff87', edgecolor=DARK_GRID, linewidth=1.2)),
    (ax.bar(x,         attn_rest, width, label='Attention Only', color='#00d4ff', edgecolor=DARK_GRID, linewidth=1.2)),
    (ax.bar(x + width, ffn_rest,  width, label='FFN Only',       color='#ff006e', edgecolor=DARK_GRID, linewidth=1.2)),
]
for bars in bars_sets:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.1%}',
                ha='center', va='bottom', fontsize=9, color=TEXT_PRIMARY, fontweight='bold')

ax.set_xlabel('Layer', fontsize=14, labelpad=10)
ax.set_ylabel('Restoration Score (F1 Macro)', fontsize=14, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels([f'Layer {l}' for l in top_layers], fontsize=13, fontweight='bold')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')
ax.legend(fontsize=13, loc='upper right', framealpha=0.15,
          edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY, fancybox=True)
ax.set_title('COMPONENT-LEVEL CAUSAL RESTORATION\n'
             'Attention vs FFN: where does the information live within each layer?',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
plt.savefig(os.path.join(RESULTS_DIR, 'activation_patching_03_component_bars.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()

## Part D — Per-Emotion Causal Profiles

### D.1 — Restoration Curves for Representative Emotions

In [ ]:
emotion_gap = baseline_per_emotion - compressed_per_emotion
valid_gaps = sorted([(i, emotion_gap[i]) for i in valid_indices], key=lambda x: -x[1])

fragile = [emotion_names[idx] for idx, _ in valid_gaps[:4]]
robust_candidates = [(idx, gap) for idx, gap in valid_gaps if gap > GAP_EPSILON]
robust = [emotion_names[idx] for idx, _ in robust_candidates[-4:]]

selected_emotions, seen = [], set()
for emo in fragile + robust:
    if emo not in seen:
        selected_emotions.append(emo)
        seen.add(emo)
selected_emotions = selected_emotions[:8]

print('Selected emotions for causal profiles:')
for emo in selected_emotions:
    idx = emotion_names.index(emo)
    print(f'  {emo:18s}: baseline={baseline_per_emotion[idx]:.4f}  '
          f'compressed={compressed_per_emotion[idx]:.4f}  gap={emotion_gap[idx]:.4f}')

fig, ax = plt.subplots(figsize=(16, 8))
palette = ['#ff006e', '#ff4d6d', '#ff9500', '#ffd60a',
           '#00ff87', '#00d4ff', '#7b2ff7', '#b388ff']

for i, emo in enumerate(selected_emotions):
    emo_idx = emotion_names.index(emo)
    curve = compute_restoration(layer_patched_f1[:, emo_idx],
                                 compressed_per_emotion[emo_idx],
                                 baseline_per_emotion[emo_idx])
    color = palette[i % len(palette)]
    ax.plot(range(N_ENCODER_LAYERS), curve, '-o', color=color, linewidth=2.5,
            markersize=7, label=emo, alpha=0.9,
            path_effects=[pe.withStroke(linewidth=4, foreground=color + '30')])
    peak_layer = int(np.argmax(curve))
    if curve[peak_layer] > 0.05:
        ax.annotate(f'L{peak_layer}', xy=(peak_layer, curve[peak_layer]),
                    xytext=(5, 8), textcoords='offset points',
                    fontsize=9, color=color, fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

ax.set_xlabel('Encoder Layer', fontsize=14, labelpad=10)
ax.set_ylabel('Restoration Score', fontsize=14, labelpad=10)
ax.set_xticks(range(N_ENCODER_LAYERS))
ax.set_xticklabels([f'L{i}' for i in range(N_ENCODER_LAYERS)], fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, 1.1)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')
ax.legend(fontsize=11, loc='upper left', framealpha=0.15,
          edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY, fancybox=True, ncol=2)
ax.set_title('PER-EMOTION CAUSAL PROFILES\n'
             'Restoration curves reveal where each emotion\'s information concentrates',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
plt.savefig(os.path.join(RESULTS_DIR, 'activation_patching_04_per_emotion_curves.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()

### D.2 — Causal Brain Scan (2 x 2 Archetypes)

In [ ]:
max_restoration_per_emotion = restoration_valid.max(axis=0)
mean_restoration_per_emotion = restoration_valid.mean(axis=0)

idx_localized = int(np.argmax(max_restoration_per_emotion))
spread = mean_restoration_per_emotion.copy()
spread[idx_localized] = -1
idx_distributed = int(np.argmax(spread))
gaps_valid = [emotion_gap[valid_indices[i]] for i in range(n_valid)]
gaps_copy = list(gaps_valid)
gaps_copy[idx_localized] = -1
gaps_copy[idx_distributed] = -1
idx_fragile = int(np.argmax(gaps_copy))
robust_candidates = [(i, mean_restoration_per_emotion[i]) for i in range(n_valid)
                     if i not in (idx_localized, idx_distributed, idx_fragile)
                     and max_restoration_per_emotion[i] > 0.01]
idx_robust = min(robust_candidates, key=lambda x: x[1])[0] if robust_candidates else 0

brain_scan_indices = [idx_localized, idx_fragile, idx_distributed, idx_robust]
brain_scan_labels  = ['LOCALIZED', 'FRAGILE', 'DISTRIBUTED', 'ROBUST']
brain_scan_colors  = ['#ff006e', '#ff9500', '#00d4ff', '#00ff87']

print('Selected emotions for brain scan:')
for vidx, label in zip(brain_scan_indices, brain_scan_labels):
    print(f'  {label:12s}: {valid_emotions[vidx]} '
          f'(peak restoration = {max_restoration_per_emotion[vidx]:.3f})')

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for ax, vidx, label, accent in zip(axes.flat, brain_scan_indices, brain_scan_labels, brain_scan_colors):
    emo_name = valid_emotions[vidx]
    emo_full_idx = valid_indices[vidx]
    grid = np.zeros((N_ENCODER_LAYERS, 3))  # Full, Attention, FFN

    full_curve = compute_restoration(layer_patched_f1[:, emo_full_idx],
                                     compressed_per_emotion[emo_full_idx],
                                     baseline_per_emotion[emo_full_idx])
    grid[:, 0] = full_curve
    for l in range(N_ENCODER_LAYERS):
        if (l, 'attention') in component_patched:
            a_r = compute_restoration(component_patched[(l, 'attention')][0][emo_full_idx],
                                      compressed_per_emotion[emo_full_idx],
                                      baseline_per_emotion[emo_full_idx])
            grid[l, 1] = a_r
        if (l, 'ffn') in component_patched:
            f_r = compute_restoration(component_patched[(l, 'ffn')][0][emo_full_idx],
                                      compressed_per_emotion[emo_full_idx],
                                      baseline_per_emotion[emo_full_idx])
            grid[l, 2] = f_r

    im = ax.imshow(grid, cmap=RESTORATION_CMAP, vmin=0, vmax=1,
                   aspect='auto', interpolation='nearest')
    ax.set_yticks(range(N_ENCODER_LAYERS))
    ax.set_yticklabels([f'L{i}' for i in range(N_ENCODER_LAYERS)], fontsize=9)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['Full Layer', 'Attention', 'FFN'], fontsize=11, fontweight='bold')
    ax.get_xticklabels()[0].set_color('#00ff87')
    ax.get_xticklabels()[1].set_color('#00d4ff')
    ax.get_xticklabels()[2].set_color('#ff006e')

    for row in range(N_ENCODER_LAYERS):
        for col in range(3):
            val = grid[row, col]
            if val > 0.02:
                txt_color = '#000000' if val > 0.6 else TEXT_PRIMARY
                ax.text(col, row, f'{val:.0%}', ha='center', va='center',
                        fontsize=8, color=txt_color, fontweight='bold')

    ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, N_ENCODER_LAYERS, 1), minor=True)
    ax.grid(which='minor', color=DARK_BG, linewidth=1.5)
    ax.tick_params(which='minor', size=0)
    peak_row = int(np.argmax(grid[:, 0]))
    ax.plot(-0.3, peak_row, '>', markersize=10, color=accent, clip_on=False)

    ax.set_title(f'{emo_name.upper()}\n{label}',
                 fontsize=14, fontweight='bold', pad=12, color=accent,
                 path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

fig.suptitle('CAUSAL BRAIN SCAN\nActivation patching reveals each emotion\'s neural footprint',
             fontsize=20, fontweight='bold', y=1.01,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
fig.text(0.5, 0.975,
         'Brighter = higher restoration  |  Arrow = critical layer  |  '
         'Attention/FFN shown for top layers only',
         fontsize=10, color=TEXT_SECONDARY, ha='center',
         path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Restoration Score', fontsize=11, color=TEXT_PRIMARY)
cbar.ax.tick_params(colors=TEXT_SECONDARY)
cbar.outline.set_edgecolor(DARK_GRID)

plt.tight_layout(rect=[0, 0, 0.91, 0.96])
plt.savefig(os.path.join(RESULTS_DIR, 'activation_patching_05_brain_scan.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()

## Part E — Cross-validation vs Lesion Study

Correlate patching restoration (what comes *back*) with lesion sensitivity (what
was *lost*). If both measure the same underlying signal, the correlation should
be positive.

In [ ]:
lesion_csv = os.path.join(RESULTS_DIR, 'notebook8', 'lesion_per_layer.csv')
if not os.path.exists(lesion_csv):
    lesion_csv = os.path.join(RESULTS_DIR, 'lesion_per_layer.csv')

patching_vs_lesion_df = None
r_pearson = None
r_spearman = None
p_pearson = None
p_spearman = None

if os.path.exists(lesion_csv):
    df_lesion = pd.read_csv(lesion_csv)
    print(f'Loaded {len(df_lesion)} lesion rows from {lesion_csv}')

    lesion_sensitivity = np.zeros((N_ENCODER_LAYERS, num_labels))
    for _, row in df_lesion.iterrows():
        l = int(row['layer'])
        if row['emotion'] in emotion_names and pd.notna(row.get('retention', np.nan)):
            lesion_sensitivity[l, emotion_names.index(row['emotion'])] = 1.0 - row['retention']

    lesion_flat = lesion_sensitivity[:, valid_indices].flatten()
    patching_flat = restoration_valid.flatten()

    r_pearson, p_pearson = pearsonr(lesion_flat, patching_flat)
    r_spearman, p_spearman = spearmanr(lesion_flat, patching_flat)

    patching_vs_lesion_df = pd.DataFrame({
        'lesion_sensitivity': lesion_flat,
        'patching_restoration': patching_flat,
    })

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.scatter(lesion_flat, patching_flat, s=15, alpha=0.5, color='#00d4ff', edgecolor='none')
    mask = (lesion_flat > 0) | (patching_flat > 0)
    if mask.sum() > 2:
        z = np.polyfit(lesion_flat[mask], patching_flat[mask], 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, lesion_flat.max(), 100)
        ax.plot(x_line, p(x_line), '--', color='#ff006e', linewidth=2, alpha=0.7)

    ax.set_xlabel('Lesion Sensitivity (1 - retention)', fontsize=14, labelpad=10)
    ax.set_ylabel('Patching Restoration Score', fontsize=14, labelpad=10)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.grid(True, alpha=0.08, color='#ffffff')
    ax.text(0.05, 0.95,
            f'Pearson r = {r_pearson:.3f} (p = {p_pearson:.2e})\n'
            f'Spearman r = {r_spearman:.3f} (p = {p_spearman:.2e})',
            transform=ax.transAxes, fontsize=12, color=TEXT_PRIMARY,
            verticalalignment='top', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor=DARK_GRID, alpha=0.8))
    ax.set_title('PATCHING vs LESION VALIDATION\n'
                 'Do layers that cause most damage when compressed also restore most when patched?',
                 fontsize=16, fontweight='bold', pad=15,
                 path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
    plt.savefig(os.path.join(RESULTS_DIR, 'activation_patching_06_validation.png'),
                dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
    plt.show()
    print(f'\nCorrelation: Pearson r = {r_pearson:.3f}, Spearman r = {r_spearman:.3f}')
else:
    print(f'Lesion results not found (tried results/notebook8/ and results/).')
    print('Run notebook 08 first to enable this cross-validation.')

## Part F — Export

All results as CSV under `results/notebook5/`:

- `activation_patching_per_layer.csv` — long form (layer x emotion) with F1s and restoration.
- `activation_patching_per_component.csv` — long form (layer x component x emotion).
- `activation_patching_summary.csv` — macro F1 for baseline, compressed, and every patched variant.
- `restoration_matrix.csv` — wide matrix (layers x emotions) of restoration scores.
- `mean_restoration_per_layer.csv` — mean restoration per layer across valid emotions.
- `critical_layer_per_emotion.csv` — per-emotion critical layer + peak restoration + concentration.
- `baseline_vs_compressed.csv` — per-emotion F1 reference (baseline, compressed, gap).
- `patching_vs_lesion.csv` — scatter data used for Part E correlation (if lesion exists).
- `patching_master_summary.csv` — one-row summary.

In [ ]:
exports = {}

# 1. Per-layer (long)
rows_layer = []
for layer_idx in range(N_ENCODER_LAYERS):
    for i, emo in enumerate(emotion_names):
        rest = float(compute_restoration(np.array([layer_patched_f1[layer_idx, i]]),
                                         compressed_per_emotion[i],
                                         baseline_per_emotion[i])[0])
        rows_layer.append({
            'layer': layer_idx,
            'emotion': emo,
            'baseline_f1': baseline_per_emotion[i],
            'compressed_f1': compressed_per_emotion[i],
            'patched_f1': layer_patched_f1[layer_idx, i],
            'restoration_score': rest,
        })
exports['activation_patching_per_layer.csv'] = pd.DataFrame(rows_layer)

# 2. Per-component (long)
rows_comp = []
for (layer_idx, component), (per_emo, _) in component_patched.items():
    for i, emo in enumerate(emotion_names):
        rest = float(compute_restoration(np.array([per_emo[i]]),
                                         compressed_per_emotion[i],
                                         baseline_per_emotion[i])[0])
        rows_comp.append({
            'layer': layer_idx,
            'component': component,
            'emotion': emo,
            'baseline_f1': baseline_per_emotion[i],
            'compressed_f1': compressed_per_emotion[i],
            'patched_f1': per_emo[i],
            'restoration_score': rest,
        })
exports['activation_patching_per_component.csv'] = pd.DataFrame(rows_comp)

# 3. Summary (macro)
rows_summary = [
    {'model': 'baseline',                           'f1_macro': baseline_f1_macro},
    {'model': f'compressed_rank_{PATCH_RANK}',     'f1_macro': compressed_f1_macro},
]
for layer_idx in range(N_ENCODER_LAYERS):
    rows_summary.append({'model': f'patched_layer_{layer_idx}',
                         'f1_macro': layer_patched_macro[layer_idx]})
for (layer_idx, component), (_, f1_m) in component_patched.items():
    rows_summary.append({'model': f'patched_layer_{layer_idx}_{component}',
                         'f1_macro': f1_m})
exports['activation_patching_summary.csv'] = pd.DataFrame(rows_summary)

# 4. Restoration matrix (wide)
restoration_df = pd.DataFrame(
    restoration_matrix,
    index=[f'L{i}' for i in range(N_ENCODER_LAYERS)],
    columns=emotion_names,
)
restoration_df.index.name = 'layer'
exports['restoration_matrix.csv'] = restoration_df.reset_index()

# 5. Mean restoration per layer
exports['mean_restoration_per_layer.csv'] = pd.DataFrame({
    'layer': range(N_ENCODER_LAYERS),
    'mean_restoration_valid': mean_restoration_per_layer,
    'mean_restoration_all': restoration_matrix.mean(axis=1),
    'layer_patched_f1_macro': layer_patched_macro,
})

# 6. Critical layer per emotion
rows_critical = []
for vi in range(n_valid):
    curve = restoration_valid[:, vi]
    peak = int(np.argmax(curve))
    total = curve.sum()
    concentration = curve[peak] / total if total > 1e-8 else 0.0
    rows_critical.append({
        'emotion': valid_emotions[vi],
        'critical_layer': peak,
        'peak_restoration': float(curve[peak]),
        'mean_restoration': float(curve.mean()),
        'concentration': float(concentration),
    })
exports['critical_layer_per_emotion.csv'] = pd.DataFrame(rows_critical)

# 7. Baseline vs compressed
exports['baseline_vs_compressed.csv'] = pd.DataFrame({
    'emotion': emotion_names,
    'baseline_f1': baseline_per_emotion,
    'compressed_f1': compressed_per_emotion,
    'gap': baseline_per_emotion - compressed_per_emotion,
    'is_valid': valid_mask,
})

# 8. Patching vs lesion
if patching_vs_lesion_df is not None:
    exports['patching_vs_lesion.csv'] = patching_vs_lesion_df

# 9. Master summary
exports['patching_master_summary.csv'] = pd.DataFrame([{
    'patch_rank': PATCH_RANK,
    'n_emotions_total': num_labels,
    'n_emotions_valid': n_valid,
    'baseline_f1_macro': baseline_f1_macro,
    'compressed_f1_macro': compressed_f1_macro,
    'best_layer_patch': int(np.argmax(layer_patched_macro)),
    'best_layer_patch_f1': float(layer_patched_macro.max()),
    'mean_restoration_global': float(mean_restoration_per_layer.mean()),
    'top_k_layers_component': ','.join(map(str, map(int, top_layers))),
    'pearson_vs_lesion': r_pearson if r_pearson is not None else float('nan'),
    'spearman_vs_lesion': r_spearman if r_spearman is not None else float('nan'),
}])

for fname, df in exports.items():
    out_path = os.path.join(EXPORT_DIR, fname)
    df.to_csv(out_path, index=False)
    print(f'  [{len(df):>5d} rows] {fname}')

print('\n' + '=' * 65)
print('ACTIVATION PATCHING STUDY COMPLETE')
print('=' * 65)
print(f'Patch rank:         {PATCH_RANK}')
print(f'Baseline F1 macro:  {baseline_f1_macro:.4f}')
print(f'Compressed F1 macro:{compressed_f1_macro:.4f}')
print(f'Best single-layer:  L{int(np.argmax(layer_patched_macro))}  '
      f'(F1 = {layer_patched_macro.max():.4f})')
print(f'\nMost critical layers (top {TOP_K_LAYERS}):')
for l in top_layers:
    print(f'  L{int(l)}: mean restoration = {mean_restoration_per_layer[int(l)]:.3f}')
print(f'\nLocalized emotions (concentration > 0.4):')
for row in exports['critical_layer_per_emotion.csv'].itertuples():
    if row.concentration > 0.4:
        print(f'  {row.emotion:18s}: L{row.critical_layer}  concentration = {row.concentration:.1%}')
print(f'\nExported {len(exports)} CSVs to {EXPORT_DIR}')

## 5.5 Ablación por corrupción: separando localización de posición

Patching parte de un modelo colapsado y restaura una capa para ver cuánto vuelve. Corrupción hace lo opuesto: parte del modelo intacto y rompe una capa para ver cuánto se pierde, neutralizando el sesgo de cercanía al clasificador.


In [ ]:
# Experimento A: corromper una capa completa a la vez (rank 64), resto intacto.
from src.compression.svd import filter_layer_names

all_target_names = get_target_layer_names(baseline_model)

corruption_per_layer_rows = []
corruption_per_layer_per_emotion_rows = []

print(f'Baseline F1 macro: {baseline_f1_macro:.4f}')
print(f'\nExperimento A — Corrupcion por capa (rank={PATCH_RANK}):')
print('-' * 68)

for layer_idx in range(N_ENCODER_LAYERS):
    torch.manual_seed(42); np.random.seed(42)
    layer_targets = filter_layer_names(all_target_names, layers=[layer_idx])
    corrupted = apply_svd_compression(baseline_model, rank=PATCH_RANK, layer_names=layer_targets)
    corrupted.eval().to(DEVICE)

    per_emo, f1m = evaluate(corrupted)
    drop = baseline_f1_macro - f1m
    retention = f1m / baseline_f1_macro * 100
    print(f'  L{layer_idx:>2d}  ({len(layer_targets)} matrices):  '
          f'F1 = {f1m:.4f}  drop = {drop:+.4f}  retention = {retention:5.1f}%')

    corruption_per_layer_rows.append({
        'layer': layer_idx,
        'f1_macro': float(f1m),
        'f1_drop': float(drop),
        'retention_pct': float(retention),
    })
    for i, emo in enumerate(emotion_names):
        corruption_per_layer_per_emotion_rows.append({
            'layer': layer_idx,
            'emotion': emo,
            'baseline_f1': float(baseline_per_emotion[i]),
            'corrupted_f1': float(per_emo[i]),
            'f1_drop': float(baseline_per_emotion[i] - per_emo[i]),
        })

    del corrupted
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

corruption_per_layer = pd.DataFrame(corruption_per_layer_rows)
corruption_per_layer_per_emotion = pd.DataFrame(corruption_per_layer_per_emotion_rows)

corruption_per_layer.to_csv(os.path.join(EXPORT_DIR, 'corruption_per_layer.csv'), index=False)
corruption_per_layer_per_emotion.to_csv(
    os.path.join(EXPORT_DIR, 'corruption_per_layer_per_emotion.csv'), index=False)
print(f'\nSaved: corruption_per_layer.csv ({len(corruption_per_layer)} rows)')
print(f'Saved: corruption_per_layer_per_emotion.csv ({len(corruption_per_layer_per_emotion)} rows)')

In [ ]:
# Experimento B: corromper attention vs FFN solo en capas tardias 8-11.
late_layers = [8, 9, 10, 11]
components = ['attention', 'ffn']

corruption_per_component_rows = []
print(f'\nExperimento B — Corrupcion por componente (rank={PATCH_RANK}):')
print('-' * 68)

for layer_idx in late_layers:
    for comp in components:
        torch.manual_seed(42); np.random.seed(42)
        targets = filter_layer_names(all_target_names, component=comp, layers=[layer_idx])
        corrupted = apply_svd_compression(baseline_model, rank=PATCH_RANK, layer_names=targets)
        corrupted.eval().to(DEVICE)

        per_emo, f1m = evaluate(corrupted)
        drop = baseline_f1_macro - f1m
        retention = f1m / baseline_f1_macro * 100
        print(f'  L{layer_idx:>2d} {comp:>9s}  ({len(targets)} matrices):  '
              f'F1 = {f1m:.4f}  drop = {drop:+.4f}  retention = {retention:5.1f}%')

        corruption_per_component_rows.append({
            'layer': layer_idx,
            'component': comp,
            'f1_macro': float(f1m),
            'f1_drop': float(drop),
            'retention_pct': float(retention),
        })

        del corrupted
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

corruption_per_component = pd.DataFrame(corruption_per_component_rows)
corruption_per_component.to_csv(
    os.path.join(EXPORT_DIR, 'corruption_per_component.csv'), index=False)
print(f'\nSaved: corruption_per_component.csv ({len(corruption_per_component)} rows)')

In [ ]:
# Figura 1: F1 al RESTAURAR vs F1 al CORROMPER cada capa.
patching_csv_candidates = [
    os.path.join(RESULTS_DIR, 'csvs', 'notebook5', 'activation_patching_per_layer.csv'),
    os.path.join(EXPORT_DIR, 'activation_patching_per_layer.csv'),
]
patching_csv = next((p for p in patching_csv_candidates if os.path.exists(p)), None)
if patching_csv is None:
    raise FileNotFoundError('activation_patching_per_layer.csv not found in expected locations')

patching_per_layer_df = pd.read_csv(patching_csv)
patching_macro = patching_per_layer_df.groupby('layer')['patched_f1'].mean().sort_index().to_numpy()
corruption_macro = corruption_per_layer.sort_values('layer')['f1_macro'].to_numpy()
layers = np.arange(N_ENCODER_LAYERS)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(layers, patching_macro, 'o-', color='#00d4ff', linewidth=2.5, markersize=11,
        label='F1 al RESTAURAR esa capa (patching)',
        path_effects=[pe.withStroke(linewidth=4, foreground=DARK_BG)])
ax.plot(layers, corruption_macro, 's-', color='#ff006e', linewidth=2.5, markersize=11,
        label='F1 al CORROMPER esa capa (corruption)',
        path_effects=[pe.withStroke(linewidth=4, foreground=DARK_BG)])
ax.axhline(baseline_f1_macro, color='#00ff87', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Baseline F1 = {baseline_f1_macro:.3f}')

ax.set_xlabel('Encoder Layer', fontsize=14, labelpad=10)
ax.set_ylabel('F1 macro', fontsize=14, labelpad=10)
ax.set_xticks(layers)
ax.set_xticklabels([f'L{i}' for i in layers], fontsize=12, fontweight='bold')
ax.set_ylim(-0.02, max(baseline_f1_macro, corruption_macro.max(), patching_macro.max()) * 1.12)
ax.legend(fontsize=12, facecolor=DARK_BG, edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY, loc='upper left')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')

ax.set_title('PATCHING vs CORRUPTION POR CAPA\n'
             'Localización funcional vs posición terminal',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

fig_path = os.path.join(RESULTS_DIR, 'comparison_patching_vs_corruption.png')
plt.savefig(fig_path, dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# Figura 2: barras agrupadas attention vs FFN en capas tardias.
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(late_layers))
width = 0.36

attn_data = (corruption_per_component[corruption_per_component['component'] == 'attention']
             .sort_values('layer')['f1_macro'].to_numpy())
ffn_data  = (corruption_per_component[corruption_per_component['component'] == 'ffn']
             .sort_values('layer')['f1_macro'].to_numpy())

b1 = ax.bar(x - width/2, attn_data, width, color='#00d4ff',
            label='Attention corrompida', edgecolor=DARK_GRID, linewidth=1.5)
b2 = ax.bar(x + width/2, ffn_data,  width, color='#ff006e',
            label='FFN corrompida', edgecolor=DARK_GRID, linewidth=1.5)

for bars, vals in [(b1, attn_data), (b2, ffn_data)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + baseline_f1_macro * 0.01,
                f'{v:.3f}', ha='center', va='bottom',
                fontsize=11, color=TEXT_PRIMARY, fontweight='bold',
                path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

ax.axhline(baseline_f1_macro, color='#00ff87', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Baseline F1 = {baseline_f1_macro:.3f}')

ax.set_xticks(x)
ax.set_xticklabels([f'L{l}' for l in late_layers], fontsize=13, fontweight='bold')
ax.set_xlabel('Encoder Layer', fontsize=14, labelpad=10)
ax.set_ylabel('F1 macro tras corrupcion', fontsize=14, labelpad=10)
ax.legend(fontsize=12, facecolor=DARK_BG, edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY, loc='upper left')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')
ax.set_ylim(0, baseline_f1_macro * 1.12)

ax.set_title('CORRUPCIÓN POR COMPONENTE EN CAPAS TARDÍAS\n'
             'Attention vs FFN (rank 64)',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

fig_path = os.path.join(RESULTS_DIR, 'corruption_per_component.png')
plt.savefig(fig_path, dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# Resumen final: hallazgos del experimento de corrupcion + comparativa con patching.
mask_03  = corruption_per_layer['layer'].between(0, 3)
mask_811 = corruption_per_layer['layer'].between(8, 11)
drop_03  = corruption_per_layer.loc[mask_03,  'f1_drop'].mean()
drop_811 = corruption_per_layer.loc[mask_811, 'f1_drop'].mean()

idx_max_drop = corruption_per_layer['f1_drop'].idxmax()
idx_min_drop = corruption_per_layer['f1_drop'].idxmin()
row_max = corruption_per_layer.loc[idx_max_drop]
row_min = corruption_per_layer.loc[idx_min_drop]

l11_attn = corruption_per_component.query("layer == 11 and component == 'attention'").iloc[0]
l11_ffn  = corruption_per_component.query("layer == 11 and component == 'ffn'").iloc[0]
ratio_l11 = (l11_ffn['f1_drop'] / l11_attn['f1_drop']
             if abs(l11_attn['f1_drop']) > 1e-6 else float('inf'))

print('=' * 68)
print('EXPERIMENTO A (corrupcion por capa):')
print(f"- Capa con mayor caida: L_{int(row_max['layer'])} "
      f"(F1 = {row_max['f1_macro']:.3f}, drop = {-row_max['f1_drop']:+.3f})")
print(f"- Capa con menor caida: L_{int(row_min['layer'])} "
      f"(F1 = {row_min['f1_macro']:.3f}, drop = {-row_min['f1_drop']:+.3f})")
print(f'- Caida media en capas 0-3:  {-drop_03:+.3f}')
print(f'- Caida media en capas 8-11: {-drop_811:+.3f}')
print()
print('EXPERIMENTO B (corrupcion por componente):')
print(f"- FFN-L11 corrompida:       F1 = {l11_ffn['f1_macro']:.3f}  (drop = {-l11_ffn['f1_drop']:+.3f})")
print(f"- Attention-L11 corrompida: F1 = {l11_attn['f1_macro']:.3f}  (drop = {-l11_attn['f1_drop']:+.3f})")
print(f'- Ratio FFN/Attention drop en L11: {ratio_l11:.1f}x')
print('=' * 68)
print()

# Tabla comparativa patching vs corruption en capas criticas
patching_macro_by_layer = patching_per_layer_df.groupby('layer')['patched_f1'].mean()
corruption_macro_by_layer = corruption_per_layer.set_index('layer')['f1_macro']

print('Comparacion patching vs corruption en capas criticas:')
header = f"{'Capa':<6}{'F1 patching':>14}{'F1 corruption':>17}{'Interpretacion':>34}"
print(header)
print('-' * len(header))
for L in [8, 9, 10, 11]:
    fp = float(patching_macro_by_layer.loc[L])
    fc = float(corruption_macro_by_layer.loc[L])
    high_p = fp > 0.5 * baseline_f1_macro
    high_c = fc > 0.5 * baseline_f1_macro
    if high_p and not high_c:
        interp = 'Critica funcionalmente'
    elif not high_p and high_c:
        interp = 'Solo posicional (no critica)'
    elif high_p and high_c:
        interp = 'Redundante / sustituible'
    else:
        interp = 'Critica + posicional'
    print(f'L{L:<5}{fp:>14.3f}{fc:>17.3f}{interp:>34}')